# Mô hình AI: Đánh giá độ phù hợp Ứng viên - Công việc
**Phương pháp:** Word2Vec + TF-IDF + XGBoost

Notebook này thực hiện các bước:
1.  **Xử lý dữ liệu:** Làm sạch, tự động gán nhãn.
2.  **Feature Engineering:** Kết hợp Word Embeddings (Word2Vec) và thống kê từ vựng (TF-IDF).
3.  **Training:** Tinh chỉnh tham số và huấn luyện XGBoost.
4.  **Lưu trữ:** Xuất mô hình để sử dụng trong ứng dụng.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import joblib
import os

# Các thư viện Machine Learning
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix
from gensim.models import Word2Vec

# Cấu hình hiển thị
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")


## 1. Load và Làm sạch Dữ liệu

In [2]:
# Đọc dữ liệu từ file CSV đã được tổng hợp
try:
    df = pd.read_csv("../data/csv_data/aggregated_data.csv")
    # print(f"Tải dữ liệu thành công. Kích thước: {df.shape}")
    df.head()
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file dữ liệu.")
    df = pd.DataFrame()

# Loại bỏ các bản ghi rác hoặc thiếu thông tin quan trọng
if not df.empty:
    # Loại bỏ các dòng có mô tả là 'undefined'
    if 'job_post_description' in df.columns:
        df = df[df['job_post_description'] != 'undefined']
        df.dropna(subset=['job_post_description'], inplace=True)

    # Loại bỏ các dòng thiếu tiêu đề công việc
    if 'job_post_title' in df.columns:
        df = df[df['job_post_title'].str.strip() != '']
        df.dropna(subset=['job_post_title'], inplace=True)

    print(f"Kích thước sau khi làm sạch: {df.shape}")
df.head(100)


Lỗi: Không tìm thấy file dữ liệu.


""


## 2. Tự động Gán nhãn (Auto-Labeling)
Do dữ liệu chưa có nhãn 'match' (phù hợp hay không), ta giả lập nhãn dựa trên độ tương đồng về kỹ năng (Skills).

In [3]:
# Xử lý giá trị null trước khi tính toán
df['candidate_skills'] = df['candidate_skills'].fillna('')
df['job_post_skills'] = df['job_post_skills'].fillna('')

# Sử dụng TF-IDF tạm thời để tính độ tương đồng giữa kỹ năng ứng viên và yêu cầu
tfidf_temp = TfidfVectorizer()
all_skills = pd.concat([df['candidate_skills'], df['job_post_skills']])
tfidf_temp.fit(all_skills)

# Biến đổi sang vector
skill_vec_cand = tfidf_temp.transform(df['candidate_skills'])
skill_vec_job = tfidf_temp.transform(df['job_post_skills'])

# Tính Cosine Similarity cho từng cặp hàng
similarities = np.array([cosine_similarity(skill_vec_cand[i], skill_vec_job[i])[0][0] for i in range(len(df))])

# Gán nhãn: Similarity >= 0.1 là HIGH (1), ngược lại là LOW (0)
# Ngưỡng 0.1 được chọn vì trường skills thường ngắn, độ trùng lặp từ thấp
THRESHOLD = 0.1
df['match'] = (similarities >= THRESHOLD).astype(int)

print(f"Phân bố nhãn (Threshold={THRESHOLD}):")
print(df['match'].value_counts())


KeyError: 'candidate_skills'

## 3. Tiền xử lý Văn bản (NLP)

In [ ]:
# Load danh sách stopwords tiếng Việt
STOPWORDS_PATH = "../data/nlp/vietnamese-stopwords.txt"
vietnamese_stopwords = set()
try:
    with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
        vietnamese_stopwords = set(line.strip() for line in f if line.strip())
except FileNotFoundError:
    print("Warning: Không tìm thấy file stopwords.")

def preprocess_text(text):
    """
    Hàm làm sạch văn bản:
    1. Chuyển về chữ thường.
    2. Loại bỏ email, URL.
    3. Giữ lại chữ cái tiếng Việt và số, loại bỏ ký tự đặc biệt.
    4. Loại bỏ stopwords.
    """
    if not isinstance(text, str) or not text:
        return ""

    text = text.lower()
    text = re.sub(r'\S+@\S+', ' ', text) # Bỏ email
    text = re.sub(r'http\S+', ' ', text) # Bỏ URL
    # Regex giữ lại chữ cái tiếng Việt và số
    text = re.sub(r'[^a-zàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ0-9\s]', ' ', text)

    tokens = text.split()
    # Lọc từ: không phải stopword, không phải số thuần túy, độ dài > 1
    clean_tokens = [t for t in tokens if t not in vietnamese_stopwords and not t.isdigit() and len(t) > 1]

    return " ".join(clean_tokens)

# Tạo cột văn bản tổng hợp cho Ứng viên và Công việc
# Kết hợp các trường thông tin quan trọng lại thành một đoạn văn bản duy nhất
cand_cols = [c for c in ['candidate_summary', 'candidate_skills', 'candidate_education'] if c in df.columns]
job_cols = [c for c in ['job_post_title', 'job_post_description', 'job_post_industry', 'job_post_level', 'job_post_skills'] if c in df.columns]

df['candidate_full_text'] = df[cand_cols].fillna('').apply(lambda x: ' '.join(x.values.astype(str)), axis=1)
df['job_full_text'] = df[job_cols].fillna('').apply(lambda x: ' '.join(x.values.astype(str)), axis=1)

# Áp dụng tiền xử lý
print("Đang xử lý văn bản...")
df['candidate_processed'] = df['candidate_full_text'].apply(preprocess_text)
df['job_processed'] = df['job_full_text'].apply(preprocess_text)
print("Hoàn tất tiền xử lý.")


## 4. Feature Engineering: Word2Vec + TF-IDF

In [ ]:
# Chia tập dữ liệu Train/Test
X = df[['candidate_processed', 'job_processed']]
y = df['match']

# Stratify để đảm bảo tỷ lệ nhãn cân bằng giữa 2 tập
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- A. Word2Vec Embedding ---
print("Đang huấn luyện Word2Vec...")
# Gom toàn bộ văn bản để huấn luyện mô hình ngôn ngữ
all_sentences = pd.concat([X_train['candidate_processed'], X_train['job_processed']]).apply(lambda x: x.split()).tolist()

# vector_size=100: Kích thước vector biểu diễn mỗi từ
# window=5: Cửa sổ ngữ cảnh
# min_count=1: Từ xuất hiện ít nhất 1 lần cũng được học
w2v_model = Word2Vec(sentences=all_sentences, vector_size=100, window=5, min_count=1, workers=4)

def get_avg_w2v_vector(text, model):
    """Tính vector trung bình của cả câu dựa trên các từ thành phần."""
    words = text.split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    return np.zeros(model.vector_size)

# Tạo feature vector từ Word2Vec
X_train_w2v_cand = np.array([get_avg_w2v_vector(text, w2v_model) for text in X_train['candidate_processed']])
X_train_w2v_job = np.array([get_avg_w2v_vector(text, w2v_model) for text in X_train['job_processed']])
X_test_w2v_cand = np.array([get_avg_w2v_vector(text, w2v_model) for text in X_test['candidate_processed']])
X_test_w2v_job = np.array([get_avg_w2v_vector(text, w2v_model) for text in X_test['job_processed']])

# --- B. TF-IDF Vectorization ---
print("Đang thực hiện TF-IDF...")
# max_features=200: Giới hạn số lượng từ vựng quan trọng nhất để giảm chiều dữ liệu
tfidf_vec = TfidfVectorizer(max_features=200, ngram_range=(1, 2))
tfidf_vec.fit(pd.concat([X_train['candidate_processed'], X_train['job_processed']]))

X_train_tfidf_cand = tfidf_vec.transform(X_train['candidate_processed'])
X_train_tfidf_job = tfidf_vec.transform(X_train['job_processed'])
X_test_tfidf_cand = tfidf_vec.transform(X_test['candidate_processed'])
X_test_tfidf_job = tfidf_vec.transform(X_test['job_processed'])

# --- C. Kết hợp Features ---
def combine_features(w2v_c, w2v_j, tfidf_c, tfidf_j):
    """
    Kết hợp các loại đặc trưng:
    1. Vector TF-IDF của Candidate & Job
    2. Vector Word2Vec của Candidate & Job
    3. Cosine Similarity giữa Candidate và Job (theo cả W2V và TF-IDF)
    """
    # Tính độ tương đồng
    sim_w2v = np.array([cosine_similarity(w2v_c[i].reshape(1,-1), w2v_j[i].reshape(1,-1))[0][0] for i in range(len(w2v_c))])
    sim_tfidf = np.array([cosine_similarity(tfidf_c[i], tfidf_j[i])[0][0] for i in range(tfidf_c.shape[0])])

    # Chuyển đổi sang dạng sparse matrix để tối ưu bộ nhớ khi nối
    return hstack([
        tfidf_c, tfidf_j,
        csr_matrix(w2v_c), csr_matrix(w2v_j),
        csr_matrix(sim_w2v.reshape(-1, 1)),
        csr_matrix(sim_tfidf.reshape(-1, 1))
    ])

X_train_final = combine_features(X_train_w2v_cand, X_train_w2v_job, X_train_tfidf_cand, X_train_tfidf_job)
X_test_final = combine_features(X_test_w2v_cand, X_test_w2v_job, X_test_tfidf_cand, X_test_tfidf_job)

print(f"Kích thước ma trận Train cuối cùng: {X_train_final.shape}")


## 5. Huấn luyện và Tinh chỉnh Mô hình (XGBoost)

In [ ]:
# Cấu hình không gian tham số để tìm kiếm
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# Sử dụng RandomizedSearchCV để tìm tham số tốt nhất
print("Đang tinh chỉnh tham số...")
search = RandomizedSearchCV(xgb, param_grid, n_iter=10, scoring='accuracy', cv=3, n_jobs=-1, random_state=42)
search.fit(X_train_final, y_train)

best_model = search.best_estimator_
print(f"Tham số tốt nhất: {search.best_params_}")
print(f"Độ chính xác tốt nhất (CV): {search.best_score_:.4f}")


## 6. Đánh giá Mô hình

In [ ]:
# Dự đoán trên tập Test
y_pred = best_model.predict(X_test_final)

print("=== BÁO CÁO ĐÁNH GIÁ ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['LOW', 'HIGH']))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['LOW', 'HIGH'], yticklabels=['LOW', 'HIGH'])
plt.title('Confusion Matrix')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()


## 7. Lưu Mô hình

In [ ]:
save_dir = "../saved_models"
os.makedirs(save_dir, exist_ok=True)

# Lưu 3 thành phần quan trọng: Model, TF-IDF Vectorizer, Word2Vec Model
joblib.dump(best_model, os.path.join(save_dir, "suitability_model_v4.joblib"))
joblib.dump(tfidf_vec, os.path.join(save_dir, "tfidf_vectorizer_v4.joblib"))
w2v_model.save(os.path.join(save_dir, "word2vec_model_v4.model"))

print(f"Đã lưu mô hình và các thành phần tại: {save_dir}")


## 8. Demo Dự đoán

In [ ]:
def predict_match(candidate_dict, job_dict, model, tfidf, w2v):
    """Hàm dự đoán cho một cặp ứng viên - công việc mới"""
    # 1. Gộp text
    cand_text = " ".join(candidate_dict.values())
    job_text = " ".join(job_dict.values())

    # 2. Preprocess
    cand_clean = preprocess_text(cand_text)
    job_clean = preprocess_text(job_text)

    # 3. Vectorize
    # TF-IDF
    t_c = tfidf.transform([cand_clean])
    t_j = tfidf.transform([job_clean])

    # Word2Vec
    w_c = get_avg_w2v_vector(cand_clean, w2v).reshape(1, -1)
    w_j = get_avg_w2v_vector(job_clean, w2v).reshape(1, -1)

    # 4. Combine
    sim_w = cosine_similarity(w_c, w_j)[0][0]
    sim_t = cosine_similarity(t_c, t_j)[0][0]

    features = hstack([
        t_c, t_j,
        csr_matrix(w_c), csr_matrix(w_j),
        csr_matrix([[sim_w]]), csr_matrix([[sim_t]])
    ])

    # 5. Predict
    prob = model.predict_proba(features)[0]
    pred = model.predict(features)[0]

    label = "HIGH" if pred == 1 else "LOW"
    return label, prob[pred]

# Dữ liệu mẫu
sample_cand = {"summary": "Lập trình viên Java 3 năm kinh nghiệm backend", "skills": "Java Spring SQL"}
sample_job = {"title": "Java Developer", "skills": "Java Spring Boot MySQL", "desc": "Phát triển backend"}

# Load lại để test (đảm bảo file lưu hoạt động)
loaded_model = joblib.load(os.path.join(save_dir, "suitability_model_v4.joblib"))
loaded_tfidf = joblib.load(os.path.join(save_dir, "tfidf_vectorizer_v4.joblib"))
loaded_w2v = Word2Vec.load(os.path.join(save_dir, "word2vec_model_v4.model"))

lbl, conf = predict_match(sample_cand, sample_job, loaded_model, loaded_tfidf, loaded_w2v)
print(f"Kết quả dự đoán: {lbl} (Độ tin cậy: {conf:.2%})")
